In [17]:
# ═════════════════════════════════════════════════════════════════════════════════
# БЛОК 1: ИМПОРТЫ И ГЛОБАЛЬНЫЕ НАСТРОЙКИ
# ═════════════════════════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import re
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, HashingVectorizer, CountVectorizer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
import nltk
from nltk.corpus import stopwords
import joblib
from nltk.stem.snowball import RussianStemmer

# Настройка seed для воспроизводимости
seed = 42
np.random.seed(seed)

# Создание директории для результатов
os.makedirs('results', exist_ok=True)

# Загрузка stopwords для русского языка
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')
russian_stopwords = stopwords.words('russian')

print("Импорты и настройка выполнены")

Импорты и настройка выполнены


In [2]:
# ═════════════════════════════════════════════════════════════════════════════════
# БЛОК 2: ПРЕДОБРАБОТКА ТЕКСТА
# ═════════════════════════════════════════════════════════════════════════════════
def preprocess_text(text):
    """Предобработка текста для русского языка"""
    if pd.isna(text) or not isinstance(text, str):
        return ""
    
    # Приведение к нижнему регистру
    text = text.lower()
    
    # Замена технических характеристик и чисел
    text = re.sub(r'\d{4,}', ' YEAR ', text)  # года
    text = re.sub(r'\d{1,3}(?:[.,]\d+)?\s*(?:gb|гб|tb|тб|mp|мп|ghz|ггц|nm|нм|inch|дюйм)', ' TECH_SPEC ', text)
    text = re.sub(r'\b\d+\b', ' NUM ', text)
    
    # Замена денежных сумм
    text = re.sub(r'\d+\s*(?:руб|рублей|р\.)', ' MONEY ', text)
    
    # Удаление специальных символов и пунктуации, сохраняя кириллические символы
    text = re.sub(r'[^\w\sа-яё]', ' ', text)
    
    # Замена множественных пробелов
    text = re.sub(r'\s+', ' ', text)
    
    # Разделение на слова
    words = text.split()
    
    # Фильтрация stopwords и коротких слов
    filtered_words = []
    for word in words:
        if len(word) <= 1:
            continue
        if word in ['NUM', 'YEAR', 'TECH_SPEC', 'MONEY']:
            filtered_words.append(word)
        elif word not in russian_stopwords and len(word) > 2:
            filtered_words.append(word)
    
    return ' '.join(filtered_words)

# Тест предобработки
test_text = "Купил iPhone 13 Pro 256 ГБ за 100000 рублей. Экран 6.1 дюйма, батарея держит 2 дня!"
print(f"Оригинал: {test_text}")
print(f"После предобработки: {preprocess_text(test_text)}")

print("Предобработка текста настроена")

Оригинал: Купил iPhone 13 Pro 256 ГБ за 100000 рублей. Экран 6.1 дюйма, батарея держит 2 дня!
После предобработки: купил iphone NUM pro TECH_SPEC YEAR рублей экран TECH_SPEC батарея держит NUM дня
Предобработка текста настроена


In [ ]:
# ═════════════════════════════════════════════════════════════════════════════════
# БЛОК 3: ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ (С ОБОСНОВАННЫМ РАЗМЕРОМ ВЫБОРКИ)
# ═════════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ")
print("="*60)

# Загрузка данных
df = pd.read_csv('informativeness.csv')
print(f"Данные загружены. Размер: {df.shape}")


target_size = 30000
print(f"\n   Выбрано: {target_size:,} примеров")

if len(df) > target_size:
    if 'informativeness' in df.columns:
        # Стратифицированная выборка для сохранения распределения
        df_sampled, _ = train_test_split(
            df, 
            train_size=target_size, 
            stratify=df['informativeness'].round(2),  # Точная стратификация
            random_state=seed
        )
        df = df_sampled.reset_index(drop=True)
        print(f"Стратифицированная выборка: {len(df):,} отзывов")
    else:
        df = df.sample(n=target_size, random_state=seed).reset_index(drop=True)
        print(f"Случайная выборка: {len(df):,} отзывов")
else:
    print(f"ℹИспользуем все {len(df):,} отзывов (достаточно для LSTM)")

print(f"\nНовый размер датасета: {df.shape}")
print(f"Распределение информативности в выборке:")
print(df['informativeness'].describe().round(4))

# Нормализация названий колонок
df.columns = [col.lower() for col in df.columns]

# Предобработка текста
print("\n Предобработка текста...")
df['clean_review'] = df['review'].apply(preprocess_text)

# Удаление пустых отзывов
initial_count = len(df)
df = df[df['clean_review'].str.len() > 10].reset_index(drop=True)
print(f"Удалено пустых отзывов: {initial_count - len(df)}")
print(f"Осталось отзывов после очистки: {len(df):,}")

# Визуализация распределения
plt.figure(figsize=(10, 6))
sns.histplot(df['informativeness'], bins=20, kde=True)
plt.title('Распределение информативности отзывов')
plt.xlabel('Информативность')
plt.ylabel('Количество')
plt.savefig('results/informativeness_distribution.png')
plt.close()

# Подготовка данных для обучения
X = df['clean_review'].values
y = df['informativeness'].values

# Разделение на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=seed
)

print(f"\nРазмер обучающей выборки: {len(X_train):,}")
print(f"Размер тестовой выборки: {len(X_test):,}")

print("Данные загружены и подготовлены")


ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ
Данные загружены. Размер: (458433, 2)

   Расчет оптимального размера выборки:
   - Минимум для LSTM: 15,000 примеров
   - Максимум для разумного времени: 50,000 примеров
   - Выбрано: 30,000 примеров (оптимальный баланс)
   - Стратификация по информативности для сохранения распределения
Стратифицированная выборка: 30,000 отзывов

Новый размер датасета: (30000, 2)
Распределение информативности в выборке:
count    30000.0000
mean         0.4062
std          0.3482
min          0.0000
25%          0.0000
50%          0.4348
75%          0.8261
max          1.0000
Name: informativeness, dtype: float64

🧹 Предобработка текста...
Удалено пустых отзывов: 375
Осталось отзывов после очистки: 29,625

Размер обучающей выборки: 23,700
Размер тестовой выборки: 5,925
Данные загружены и подготовлены


In [ ]:
# ═════════════════════════════════════════════════════════════════════════════════
# БЛОК 4: УТИЛИТЫ ДЛЯ ОБУЧЕНИЯ И ОЦЕНКИ
# ═════════════════════════════════════════════════════════════════════════════════
def train_and_evaluate_model(pipeline, model_name, X_train, X_test, y_train, y_test):
    """Обучение и оценка модели с сохранением результатов"""
    print(f"\n{'='*60}")
    print(f" ОБУЧЕНИЕ МОДЕЛИ: {model_name}")
    print(f"{'='*60}")
    
    # Обучение модели
    print("⚡ Обучение модели...")
    pipeline.fit(X_train, y_train)
    print(" Модель успешно обучена")
    
    # Сохранение модели
    model_path = f'results/{model_name}_model.pkl'
    joblib.dump(pipeline, model_path)
    print(f" Модель сохранена в '{model_path}'")
    
    # Предсказания
    y_pred_train = pipeline.predict(X_train)
    y_pred_test = pipeline.predict(X_test)
    
    # Расчет метрик
    train_mae = mean_absolute_error(y_train, y_pred_train)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
    train_r2 = r2_score(y_train, y_pred_train)
    test_mae = mean_absolute_error(y_test, y_pred_test)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    test_r2 = r2_score(y_test, y_pred_test)
    
    # Вывод результатов
    print("\n Результаты на обучающей выборке:")
    print(f"   MAE  : {train_mae:.4f}")
    print(f"   RMSE : {train_rmse:.4f}")
    print(f"   R²   : {train_r2:.4f}")
    
    print("\n Результаты на тестовой выборке:")
    print(f"   MAE  : {test_mae:.4f}")
    print(f"   RMSE : {test_rmse:.4f}")
    print(f"   R²   : {test_r2:.4f}")
    
    # График: Scatter plot предсказаний vs реальных значений
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.scatter(y_train, y_pred_train, alpha=0.5, color='green', s=10)
    plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2)
    plt.xlabel('Реальные значения')
    plt.ylabel('Предсказанные значения')
    plt.title(f'Train: {model_name}\nR² = {train_r2:.3f}')
    
    plt.subplot(1, 2, 2)
    plt.scatter(y_test, y_pred_test, alpha=0.5, color='red', s=10)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
    plt.xlabel('Реальные значения')
    plt.ylabel('Предсказанные значения')
    plt.title(f'Test: {model_name}\nR² = {test_r2:.3f}')
    
    plt.tight_layout()
    plt.savefig(f'results/{model_name}_predictions_scatter.png')
    plt.close()
    
    return {
        'model_name': model_name,
        'r2_train': train_r2,
        'r2_test': test_r2,
        'mae_train': train_mae,
        'mae_test': test_mae,
        'rmse_train': train_rmse,
        'rmse_test': test_rmse,
        'training_time': 0, 
        'params': str(pipeline.named_steps)
    }

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════════
# БЛОК 5: ДОПОЛНИТЕЛЬНЫЕ ВЕКТОРИЗАТОРЫ ДЛЯ РУССКОГО ЯЗЫКА
# ═════════════════════════════════════════════════════════════════════════════════
class LemmaVectorizer(BaseEstimator, TransformerMixin):
    """Векторизатор с лемматизацией для русского языка"""
    def __init__(self, max_features=5000, ngram_range=(1, 2)):
        # Сохраняем параметры как атрибуты для get_params()
        self.max_features = max_features
        self.ngram_range = ngram_range
        
        # Инициализация стеммера и векторизатора
        self.stemmer = RussianStemmer()
        self.vectorizer = TfidfVectorizer(
            max_features=max_features,
            ngram_range=ngram_range,
            min_df=3,
            max_df=0.9,
            token_pattern=r'(?u)\b[а-яё]+\b'
        )
    
    def fit(self, X, y=None):
        """Обучение векторизатора с лемматизацией"""
        # Применяем лемматизацию ко всем текстам
        X_lemmatized = [' '.join([self.stemmer.stem(word) for word in str(text).lower().split()]) for text in X]
        
        # Обучаем TF-IDF векторизатор
        self.vectorizer.fit(X_lemmatized)
        return self
    
    def transform(self, X):
        """Преобразование текстов с лемматизацией"""
        # Применяем лемматизацию
        X_lemmatized = [' '.join([self.stemmer.stem(word) for word in str(text).lower().split()]) for text in X]
        
        # Преобразуем в TF-IDF матрицу
        return self.vectorizer.transform(X_lemmatized)
    
    def get_params(self, deep=True):
        return {
            'max_features': self.max_features,
            'ngram_range': self.ngram_range
        }
    
    def set_params(self, **params):
        for key, value in params.items():
            if hasattr(self, key):
                setattr(self, key, value)
        
        # Обновляем параметры внутреннего векторизатора при необходимости
        if 'max_features' in params or 'ngram_range' in params:
            self.vectorizer = TfidfVectorizer(
                max_features=self.max_features,
                ngram_range=self.ngram_range,
                min_df=3,
                max_df=0.9,
                token_pattern=r'(?u)\b[а-яё]+\b'
            )
        
        return self

print(" LemmaVectorizer успешно инициализирован с поддержкой get_params()/set_params()")

 LemmaVectorizer успешно инициализирован с поддержкой get_params()/set_params()


In [ ]:
# ═════════════════════════════════════════════════════════════════════════════════
# БЛОК 6: ТУРНИРНЫЙ ПОДХОД - ФИКСИРОВАННАЯ ВЕКТОРИЗАЦИЯ (TF-IDF)
# ═════════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("ТУРНИР 1: TF-IDF + РАЗНЫЕ РЕГРЕССОРЫ")
print("="*60)

# Базовый TF-IDF векторизатор (фиксированный для всех моделей)
base_tfidf = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 3),
    min_df=2,
    max_df=0.95,
    token_pattern=r'(?u)\b[а-яёa-z0-9]+\b'
)

# ═════════════════════════════════════════════════════════════════════════════════
# МОДЕЛЬ 6.1: TF-IDF + Gradient Boosting 
pipeline_tfidf_gb = Pipeline([
    ('tfidf', base_tfidf),
    ('regressor', GradientBoostingRegressor(
        max_depth=6,
        learning_rate=0.05,
        n_estimators=200,
        subsample=0.9,
        min_samples_leaf=2,
        validation_fraction=0.1,
        n_iter_no_change=10,
        random_state=seed
    ))
])

results_tfidf_gb = train_and_evaluate_model(
    pipeline_tfidf_gb, 'tfidf_gb', X_train, X_test, y_train, y_test
)

# ═════════════════════════════════════════════════════════════════════════════════
# МОДЕЛЬ 6.2: TF-IDF + Random Forest
pipeline_tfidf_rf = Pipeline([
    ('tfidf', base_tfidf),
    ('regressor', RandomForestRegressor(
        n_estimators=150,
        max_depth=12,
        min_samples_leaf=3,
        max_features='sqrt',
        random_state=seed,
        n_jobs=-1,
        verbose=0
    ))
])

results_tfidf_rf = train_and_evaluate_model(
    pipeline_tfidf_rf, 'tfidf_rf', X_train, X_test, y_train, y_test
)

# ═════════════════════════════════════════════════════════════════════════════════
# МОДЕЛЬ 6.3: TF-IDF + Ridge
pipeline_tfidf_ridge = Pipeline([
    ('tfidf', base_tfidf),
    ('regressor', Ridge(
        alpha=1.0,
        random_state=seed,
        max_iter=1000
    ))
])

results_tfidf_ridge = train_and_evaluate_model(
    pipeline_tfidf_ridge, 'tfidf_ridge', X_train, X_test, y_train, y_test
)

# ═════════════════════════════════════════════════════════════════════════════════
# МОДЕЛЬ 6.4: TF-IDF + SVR
pipeline_tfidf_svr = Pipeline([
    ('tfidf', base_tfidf),
    ('regressor', SVR(
        kernel='rbf',
        C=1.0,
        epsilon=0.1,
        max_iter=5000
    ))
])

results_tfidf_svr = train_and_evaluate_model(
    pipeline_tfidf_svr, 'tfidf_svr', X_train, X_test, y_train, y_test
)


🏆 ТУРНИР 1: TF-IDF + РАЗНЫЕ РЕГРЕССОРЫ

 ОБУЧЕНИЕ МОДЕЛИ: tfidf_gb
⚡ Обучение модели...
 Модель успешно обучена
 Модель сохранена в 'results/tfidf_gb_model.pkl'

 Результаты на обучающей выборке:
   MAE  : 0.1070
   RMSE : 0.1379
   R²   : 0.8431

 Результаты на тестовой выборке:
   MAE  : 0.1188
   RMSE : 0.1540
   R²   : 0.8007

 ОБУЧЕНИЕ МОДЕЛИ: tfidf_rf
⚡ Обучение модели...
 Модель успешно обучена
 Модель сохранена в 'results/tfidf_rf_model.pkl'

 Результаты на обучающей выборке:
   MAE  : 0.2247
   RMSE : 0.2632
   R²   : 0.4281

 Результаты на тестовой выборке:
   MAE  : 0.2262
   RMSE : 0.2657
   R²   : 0.4067

 ОБУЧЕНИЕ МОДЕЛИ: tfidf_ridge
⚡ Обучение модели...
 Модель успешно обучена
 Модель сохранена в 'results/tfidf_ridge_model.pkl'

 Результаты на обучающей выборке:
   MAE  : 0.1112
   RMSE : 0.1444
   R²   : 0.8280

 Результаты на тестовой выборке:
   MAE  : 0.1389
   RMSE : 0.1828
   R²   : 0.7193

 ОБУЧЕНИЕ МОДЕЛИ: tfidf_svr
⚡ Обучение модели...


/home/kotishka/BMSTU/3course/TAI/.tai/lib/python3.12/site-packages/sklearn/svm/_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=5000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


 Модель успешно обучена
 Модель сохранена в 'results/tfidf_svr_model.pkl'

 Результаты на обучающей выборке:
   MAE  : 0.0917
   RMSE : 0.1014
   R²   : 0.9152

 Результаты на тестовой выборке:
   MAE  : 0.1495
   RMSE : 0.1848
   R²   : 0.7130


In [ ]:
# ═════════════════════════════════════════════════════════════════════════════════
# БЛОК 7: ТУРНИРНЫЙ ПОДХОД - ФИКСИРОВАННЫЙ РЕГРЕССОР (GradientBoosting)
# ═════════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print(" ТУРНИР 2: РАЗНЫЕ ВЕКТОРИЗАТОРЫ + GRADIENT BOOSTING")
print("="*60)

base_gb = GradientBoostingRegressor(
    max_depth=6,
    learning_rate=0.05,
    n_estimators=200,
    subsample=0.9,
    min_samples_leaf=2,
    validation_fraction=0.1,
    n_iter_no_change=10,
    random_state=seed
)

# ═════════════════════════════════════════════════════════════════════════════════
# МОДЕЛЬ 7.1: Character-level + GB 
pipeline_char_gb = Pipeline([
    ('char_tfidf', TfidfVectorizer(
        analyzer='char',
        ngram_range=(3, 5),
        max_features=10000,
        min_df=3
    )),
    ('regressor', base_gb)
])

results_char_gb = train_and_evaluate_model(
    pipeline_char_gb, 'char_gb', X_train, X_test, y_train, y_test
)

# ═════════════════════════════════════════════════════════════════════════════════
# МОДЕЛЬ 7.2: Лемматизация + TF-IDF + GB
pipeline_lemma_gb = Pipeline([
    ('lemmatizer', LemmaVectorizer(max_features=5000, ngram_range=(1, 2))),
    ('regressor', base_gb)
])

results_lemma_gb = train_and_evaluate_model(
    pipeline_lemma_gb, 'lemma_gb', X_train, X_test, y_train, y_test
)

# ═════════════════════════════════════════════════════════════════════════════════
# МОДЕЛЬ 7.3: Hashing + GB (FastText-like подход с другим регрессором)
pipeline_hashing_gb = Pipeline([
    ('hashing', HashingVectorizer(
        n_features=2**18,
        token_pattern=r'(?u)\b\w{3,}\b',
        norm='l2',
        alternate_sign=False
    )),
    ('regressor', base_gb)
])

results_hashing_gb = train_and_evaluate_model(
    pipeline_hashing_gb, 'hashing_gb', X_train, X_test, y_train, y_test
)


🏆 ТУРНИР 2: РАЗНЫЕ ВЕКТОРИЗАТОРЫ + GRADIENT BOOSTING

 ОБУЧЕНИЕ МОДЕЛИ: char_gb
⚡ Обучение модели...
 Модель успешно обучена
 Модель сохранена в 'results/char_gb_model.pkl'

 Результаты на обучающей выборке:
   MAE  : 0.0728
   RMSE : 0.1006
   R²   : 0.9164

 Результаты на тестовой выборке:
   MAE  : 0.0877
   RMSE : 0.1250
   R²   : 0.8686

 ОБУЧЕНИЕ МОДЕЛИ: lemma_gb
⚡ Обучение модели...
 Модель успешно обучена
 Модель сохранена в 'results/lemma_gb_model.pkl'

 Результаты на обучающей выборке:
   MAE  : 0.1122
   RMSE : 0.1464
   R²   : 0.8231

 Результаты на тестовой выборке:
   MAE  : 0.1240
   RMSE : 0.1640
   R²   : 0.7739

 ОБУЧЕНИЕ МОДЕЛИ: hashing_gb
⚡ Обучение модели...
 Модель успешно обучена
 Модель сохранена в 'results/hashing_gb_model.pkl'

 Результаты на обучающей выборке:
   MAE  : 0.1062
   RMSE : 0.1368
   R²   : 0.8456

 Результаты на тестовой выборке:
   MAE  : 0.1185
   RMSE : 0.1535
   R²   : 0.8021


In [ ]:
# ═════════════════════════════════════════════════════════════════════════════════
# БЛОК 8: ГИБРИДНЫЕ МОДЕЛИ И АНСАМБЛИ 
# ═════════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print(" ТУРНИР 3: ГИБРИДНЫЕ МОДЕЛИ")
print("="*60)

# ═════════════════════════════════════════════════════════════════════════════════
# МОДЕЛЬ 8.1: Hashing + Ridge 
pipeline_hashing_ridge = Pipeline([
    ('hashing', HashingVectorizer(
        n_features=2**18,
        token_pattern=r'(?u)\b\w{3,}\b',
        norm='l2',
        alternate_sign=False
    )),
    ('regressor', Ridge(
        alpha=1.0,
        random_state=seed,
        max_iter=1000
    ))
])

results_hashing_ridge = train_and_evaluate_model(
    pipeline_hashing_ridge, 'hashing_ridge', X_train, X_test, y_train, y_test
)

# ═════════════════════════════════════════════════════════════════════════════════
# МОДЕЛЬ 8.2: TF-IDF + CatBoost
try:
    from catboost import CatBoostRegressor
    from sklearn.base import BaseEstimator, RegressorMixin
    from sklearn.utils.validation import check_is_fitted, check_X_y, check_array
    
    # ПОЛНОСТЬЮ СОВМЕСТИМЫЙ АДАПТЕР ДЛЯ CATBOOST
    class CatBoostEstimator(BaseEstimator, RegressorMixin):
        """Полностью совместимый с sklearn эстиматор для CatBoost"""
        
        def __init__(self, 
                     iterations=300,
                     learning_rate=0.05,
                     depth=6,
                     l2_leaf_reg=3.0,
                     early_stopping_rounds=50,
                     random_state=42,
                     verbose=False,
                     eval_metric='MAE',
                     **kwargs):
            """
            Инициализация CatBoost эстиматора с полной совместимостью sklearn
            """
            self.iterations = iterations
            self.learning_rate = learning_rate
            self.depth = depth
            self.l2_leaf_reg = l2_leaf_reg
            self.early_stopping_rounds = early_stopping_rounds
            self.random_state = random_state
            self.verbose = verbose
            self.eval_metric = eval_metric
            self.kwargs = kwargs
            self.model_ = None
            self.is_fitted_ = False
        
        def fit(self, X, y, **fit_params):
            """Полностью совместимый метод fit"""
            # Проверка входных данных
            X, y = check_X_y(X, y, accept_sparse=True)
            
            # Создание и обучение модели
            self.model_ = CatBoostRegressor(
                iterations=self.iterations,
                learning_rate=self.learning_rate,
                depth=self.depth,
                l2_leaf_reg=self.l2_leaf_reg,
                early_stopping_rounds=self.early_stopping_rounds,
                random_state=self.random_state,
                verbose=self.verbose if self.verbose else False,
                eval_metric=self.eval_metric,
                **self.kwargs
            )
            
            # Обучение с поддержкой fit_params (например, eval_set)
            if 'eval_set' in fit_params:
                self.model_.fit(X, y, eval_set=fit_params['eval_set'], **fit_params)
            else:
                self.model_.fit(X, y, **fit_params)
            
            self.is_fitted_ = True
            return self
        
        def predict(self, X):
            """Полностью совместимый метод predict"""
            check_is_fitted(self, 'is_fitted_')
            X = check_array(X, accept_sparse=True)
            return self.model_.predict(X)
        
        def get_params(self, deep=True):
            """Получение параметров для GridSearchCV и Pipeline"""
            params = {
                'iterations': self.iterations,
                'learning_rate': self.learning_rate,
                'depth': self.depth,
                'l2_leaf_reg': self.l2_leaf_reg,
                'early_stopping_rounds': self.early_stopping_rounds,
                'random_state': self.random_state,
                'verbose': self.verbose,
                'eval_metric': self.eval_metric
            }
            params.update(self.kwargs)
            return params
        
        def set_params(self, **params):
            """Установка параметров"""
            for key, value in params.items():
                if hasattr(self, key):
                    setattr(self, key, value)
                elif key in self.kwargs:
                    self.kwargs[key] = value
                else:
                    self.kwargs[key] = value
            return self
        
        @property
        def named_steps(self):
            """Эмуляция атрибута named_steps для совместимости с Pipeline"""
            return {
                'tfidf': 'TfidfVectorizer(max_features=5000, ngram_range=(1, 2))',
                'regressor': f'CatBoostRegressor(iterations={self.iterations}, depth={self.depth})'
            }
        
        def __repr__(self):
            """Человекочитаемое представление"""
            return (f"CatBoostEstimator(iterations={self.iterations}, "
                    f"depth={self.depth}, learning_rate={self.learning_rate})")

    pipeline_catboost = Pipeline([
        ('tfidf', TfidfVectorizer(
            max_features=5000,  
            ngram_range=(1, 2),
            min_df=3,
            max_df=0.9,
            token_pattern=r'(?u)\b[а-яёa-z0-9]+\b'
        )),
        ('regressor', CatBoostEstimator(
            iterations=300,
            learning_rate=0.05,
            depth=6,
            l2_leaf_reg=3.0,
            early_stopping_rounds=50,
            random_state=seed,
            verbose=False,
            eval_metric='MAE'
        ))
    ])

    print(" Обучение CatBoost модели...")
    results_catboost = train_and_evaluate_model(
        pipeline_catboost, 'catboost', X_train, X_test, y_train, y_test
    )
    
    # Сохранение отдельных компонентов для production
    joblib.dump(pipeline_catboost.named_steps['tfidf'], 'results/catboost_tfidf_vectorizer.pkl')
    joblib.dump(pipeline_catboost.named_steps['regressor'].model_, 'results/catboost_model_direct.pkl')
    print(" CatBoost модель и векторизатор сохранены")
    
except ImportError:
    print(" CatBoost не установлен. Установите командой: pip install catboost")
    results_catboost = None
except Exception as e:
    print(f" Ошибка при работе с CatBoost: {e}")
    print(" Попробуйте обновить пакеты: pip install --upgrade catboost scikit-learn")
    import traceback
    traceback.print_exc()
    results_catboost = None


🏆 ТУРНИР 3: ГИБРИДНЫЕ МОДЕЛИ

 ОБУЧЕНИЕ МОДЕЛИ: hashing_ridge
⚡ Обучение модели...
 Модель успешно обучена
 Модель сохранена в 'results/hashing_ridge_model.pkl'

 Результаты на обучающей выборке:
   MAE  : 0.1057
   RMSE : 0.1378
   R²   : 0.8433

 Результаты на тестовой выборке:
   MAE  : 0.1364
   RMSE : 0.1790
   R²   : 0.7307
🚀 Обучение CatBoost модели (полностью совместимая версия)...

 ОБУЧЕНИЕ МОДЕЛИ: catboost
⚡ Обучение модели...
 Модель успешно обучена
 Модель сохранена в 'results/catboost_model.pkl'

 Результаты на обучающей выборке:
   MAE  : 0.1118
   RMSE : 0.1454
   R²   : 0.8256

 Результаты на тестовой выборке:
   MAE  : 0.1179
   RMSE : 0.1536
   R²   : 0.8016
✅ CatBoost модель и векторизатор сохранены для production


In [ ]:
# ═════════════════════════════════════════════════════════════════════════════════
# БЛОК 9: НЕЙРОСЕТЕВОЙ ПОДХОД - LSTM МОДЕЛЬ
# ═════════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print(" БЛОК 9: НЕЙРОСЕТЕВОЙ ПОДХОД - LSTM МОДЕЛЬ")
print("="*60)

# Глобальные переменные для хранения результатов LSTM
lstm_results = None
lstm_model = None
tokenizer = None

# Проверяем наличие TensorFlow/Keras
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    from tensorflow.keras.optimizers import Adam
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    from tensorflow.keras.regularizers import l2
    
    print(f" TensorFlow {tf.__version__} доступен")
    print(f" GPU доступен: {tf.config.list_physical_devices('GPU')}")
    
    # === ПОДГОТОВКА ДАННЫХ ДЛЯ LSTM ===
    print("\n" + "-"*50)
    print(" ПОДГОТОВКА ДАННЫХ ДЛЯ LSTM")
    print("-"*50)
    
    # Параметры для LSTM
    MAX_WORDS = 20000          # Размер словаря
    MAX_SEQUENCE_LENGTH = 200  # Максимальная длина последовательности
    EMBEDDING_DIM = 128        # Размерность эмбеддингов
    
    print(f" Параметры LSTM:")
    print(f"   - Максимальное количество слов: {MAX_WORDS:,}")
    print(f"   - Максимальная длина последовательности: {MAX_SEQUENCE_LENGTH}")
    print(f"   - Размерность эмбеддингов: {EMBEDDING_DIM}")
    
    # Создание токенизатора
    print("\n🔤 Токенизация текстов...")
    tokenizer = Tokenizer(
        num_words=MAX_WORDS,
        filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n',
        lower=True,
        split=' ',
        char_level=False,
        oov_token='<OOV>'
    )
    
    # Обучение токенизатора
    tokenizer.fit_on_texts(X_train)
    
    word_index = tokenizer.word_index
    vocab_size = min(MAX_WORDS, len(word_index) + 1)
    
    print(f" Статистика токенизации:")
    print(f"   - Размер словаря: {len(word_index):,} уникальных слов")
    print(f"   - Используемый словарь: {vocab_size:,} слов")
    
    # Преобразование текстов в последовательности
    X_train_seq = tokenizer.texts_to_sequences(X_train)
    X_test_seq = tokenizer.texts_to_sequences(X_test)
    
    # Паддинг
    X_train_pad = pad_sequences(
        X_train_seq, 
        maxlen=MAX_SEQUENCE_LENGTH, 
        padding='post', 
        truncating='post'
    )
    
    X_test_pad = pad_sequences(
        X_test_seq, 
        maxlen=MAX_SEQUENCE_LENGTH, 
        padding='post', 
        truncating='post'
    )
    
    print(f" Форма обучающих данных: {X_train_pad.shape}")
    print(f" Форма тестовых данных: {X_test_pad.shape}")
    
    # === СОЗДАНИЕ LSTM МОДЕЛИ ===
    print("\n" + "-"*50)
    print(" СОЗДАНИЕ LSTM МОДЕЛИ")
    print("-"*50)
    
    tf.keras.backend.clear_session()
    
    lstm_model = Sequential(name='LSTM_Informativeness_Predictor')
    
    lstm_model.add(Embedding(
        input_dim=vocab_size,
        output_dim=EMBEDDING_DIM,
        mask_zero=True,
        name='embedding_layer'
    ))
    
    lstm_model.add(Bidirectional(
        LSTM(128, 
             return_sequences=True, 
             dropout=0.3, 
             recurrent_dropout=0.3,
             kernel_regularizer=l2(0.001),
             name='lstm_layer_1'
        )
    ))
    
    lstm_model.add(Bidirectional(
        LSTM(64, 
             return_sequences=False,
             dropout=0.3, 
             recurrent_dropout=0.3,
             kernel_regularizer=l2(0.001),
             name='lstm_layer_2'
        )
    ))
    
    # Полносвязные слои
    lstm_model.add(Dense(64, activation='relu', kernel_regularizer=l2(0.01), name='dense_1'))
    lstm_model.add(Dropout(0.4, name='dropout_1'))
    lstm_model.add(Dense(32, activation='relu', kernel_regularizer=l2(0.001), name='dense_2'))
    lstm_model.add(Dropout(0.3, name='dropout_2'))
    lstm_model.add(Dense(1, activation='sigmoid', name='output_layer'))
    
    # Компиляция
    lstm_model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae', tf.keras.metrics.RootMeanSquaredError(name='rmse')]
    )
    
    print("\n Форсированное построение модели...")
    lstm_model.build(input_shape=(None, MAX_SEQUENCE_LENGTH))
    
    print("\n Архитектура LSTM модели (успешно построена):")
    lstm_model.summary()
    
    total_params = lstm_model.count_params()
    print(f"\n Общее количество параметров: {total_params:,}")
    
    # === ОБУЧЕНИЕ И ОСТАЛЬНЫЕ ШАГИ ===
    early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=8,
        restore_best_weights=True,
        verbose=1
    )
    
    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
    
    print("\n⚡ Обучение LSTM модели...")
    history = lstm_model.fit(
        X_train_pad, y_train,
        batch_size=64,
        epochs=30,
        validation_split=0.1,
        callbacks=[early_stopping, reduce_lr],
        verbose=1
    )
    
    print(" Обучение LSTM модели завершено!")
    
    # ПРЕДСКАЗАНИЯ И ОЦЕНКА
    y_pred_train_lstm = lstm_model.predict(X_train_pad, verbose=0).flatten()
    y_pred_test_lstm = lstm_model.predict(X_test_pad, verbose=0).flatten()
    
    train_mae_lstm = mean_absolute_error(y_train, y_pred_train_lstm)
    test_mae_lstm = mean_absolute_error(y_test, y_pred_test_lstm)
    train_rmse_lstm = np.sqrt(mean_squared_error(y_train, y_pred_train_lstm))
    test_rmse_lstm = np.sqrt(mean_squared_error(y_test, y_pred_test_lstm))
    train_r2_lstm = r2_score(y_train, y_pred_train_lstm)
    test_r2_lstm = r2_score(y_test, y_pred_test_lstm)
    
    print(f"\n Результаты LSTM:")
    print(f"   Train R²: {train_r2_lstm:.4f}")
    print(f"   Test R² : {test_r2_lstm:.4f}")
    print(f"   Test MAE: {test_mae_lstm:.4f}")
    
    # СОХРАНЕНИЕ
    lstm_model.save('results/lstm_model.keras')
    import pickle
    with open('results/lstm_tokenizer.pkl', 'wb') as f:
        pickle.dump(tokenizer, f)
    
    lstm_results = {
        'model_name': 'lstm',
        'r2_train': train_r2_lstm,
        'r2_test': test_r2_lstm,
        'mae_train': train_mae_lstm,
        'mae_test': test_mae_lstm,
        'rmse_train': train_rmse_lstm,
        'rmse_test': test_rmse_lstm,
        'training_time': 0,
        'params': f'LSTM: vocab={vocab_size}, max_len={MAX_SEQUENCE_LENGTH}'
    }
    
    print("\n LSTM модель успешно обучена, оценена и сохранена!")
    
except Exception as e:
    print(f" КРИТИЧЕСКАЯ ОШИБКА: {e}")
    import traceback
    traceback.print_exc()
    lstm_results = None
    # Создаем заглушку для продолжения работы
    lstm_results = {
        'model_name': 'lstm',
        'r2_train': 0.0,
        'r2_test': 0.0,
        'mae_train': 0.0,
        'mae_test': 0.0,
        'rmse_train': 0.0,
        'rmse_test': 0.0,
        'training_time': 0,
        'params': f'Ошибка обучения: {str(e)}'
    }

print("\n" + "="*60)
print(" БЛОК 9: LSTM МОДЕЛЬ ЗАВЕРШЕН")
print("="*60)


🤖 БЛОК 9: НЕЙРОСЕТЕВОЙ ПОДХОД - LSTM МОДЕЛЬ (УНИВЕРСАЛЬНАЯ ВЕРСИЯ)
✅ TensorFlow 2.20.0 доступен
📊 GPU доступен: []

--------------------------------------------------
🔧 ПОДГОТОВКА ДАННЫХ ДЛЯ LSTM
--------------------------------------------------
⚙️ Параметры LSTM:
   - Максимальное количество слов: 20,000
   - Максимальная длина последовательности: 200
   - Размерность эмбеддингов: 128

🔤 Токенизация текстов...
📈 Статистика токенизации:
   - Размер словаря: 70,608 уникальных слов
   - Используемый словарь: 20,000 слов
✅ Форма обучающих данных: (23700, 200)
✅ Форма тестовых данных: (5925, 200)

--------------------------------------------------
🏗️ СОЗДАНИЕ LSTM МОДЕЛИ (УНИВЕРСАЛЬНЫЙ ПОДХОД)
--------------------------------------------------

🔧 Форсированное построение модели...

📋 Архитектура LSTM модели (успешно построена):


Model: "LSTM_Informativeness_Predictor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_layer (Embedding)     │ (None, 200, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 200, 256)       │       263,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,997,889 (11.44 MB)

 Trainable params: 2,997,889 (11.44 MB)

 Non-trainable params: 0 (0.00 B)


✅ Общее количество параметров: 2,997,889

⚡ Обучение LSTM модели...
Epoch 1/30
334/334 ━━━━━━━━━━━━━━━━━━━━ 283s 815ms/step - loss: 0.3010 - mae: 0.2013 - rmse: 0.2446 - val_loss: 0.0479 - val_mae: 0.1593 - val_rmse: 0.1855 - learning_rate: 0.0010
Epoch 2/30
334/334 ━━━━━━━━━━━━━━━━━━━━ 273s 817ms/step - loss: 0.0438 - mae: 0.1400 - rmse: 0.1784 - val_loss: 0.0422 - val_mae: 0.1379 - val_rmse: 0.1791 - learning_rate: 0.0010
Epoch 3/30
334/334 ━━━━━━━━━━━━━━━━━━━━ 270s 808ms/step - loss: 0.0405 - mae: 0.1291 - rmse: 0.1685 - val_loss: 0.0440 - val_mae: 0.1321 - val_rmse: 0.1758 - learning_rate: 0.0010
Epoch 4/30
334/334 ━━━━━━━━━━━━━━━━━━━━ 267s 800ms/step - loss: 0.0472 - mae: 0.1360 - rmse: 0.1775 - val_loss: 0.0555 - val_mae: 0.1426 - val_rmse: 0.1854 - learning_rate: 0.0010
Epoch 5/30
334/334 ━━━━━━━━━━━━━━━━━━━━ 270s 808ms/step - loss: 0.0425 - mae: 0.1280 - rmse: 0.1675 - val_loss: 0.0399 - val_mae: 0.1242 - val_rmse: 0.1722 - learning_rate: 0.0010
Epoch 6/30
334/334 ━━━━━━━━━━━━

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════════
# ДОПОЛНИТЕЛЬНЫЙ БЛОК: ВОССТАНОВЛЕНИЕ LSTM РЕЗУЛЬТАТОВ
# ═════════════════════════════════════════════════════════════════════════════════

print("\n" + "="*70)
print("🔍 ВОССТАНОВЛЕНИЕ LSTM РЕЗУЛЬТАТОВ")
print("="*70)

# Проверка наличия сохраненных файлов
import os
import pickle
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

lstm_files = [
    'results/lstm_model.keras',
    'results/lstm_tokenizer.pkl'
]

missing_files = [f for f in lstm_files if not os.path.exists(f)]
if missing_files:
    print(f" Отсутствуют файлы: {missing_files}")
    print(" LSTM результаты не могут быть восстановлены")
    results_lstm = None
else:
    try:
        print(" Все необходимые файлы найдены")
        
        # 1. Загрузка токенизатора
        print("\n Загрузка токенизатора...")
        with open('results/lstm_tokenizer.pkl', 'rb') as f:
            tokenizer = pickle.load(f)
        
        # 2. Загрузка модели
        print(" Загрузка LSTM модели...")
        lstm_model = load_model('results/lstm_model.keras')
        print(" LSTM модель успешно загружена")
        
        # 3. Подготовка данных (восстанавливаем параметры из токенизатора)
        print("\n Подготовка данных для предсказаний...")
        
        # Восстанавливаем параметры из токенизатора и контекста
        MAX_SEQUENCE_LENGTH = 200  # Используем то же значение, что и при обучении
        
        # Преобразование текстов в последовательности
        X_train_seq = tokenizer.texts_to_sequences(X_train)
        X_test_seq = tokenizer.texts_to_sequences(X_test)
        
        # Паддинг
        X_train_pad = pad_sequences(
            X_train_seq, 
            maxlen=MAX_SEQUENCE_LENGTH, 
            padding='post', 
            truncating='post'
        )
        
        X_test_pad = pad_sequences(
            X_test_seq, 
            maxlen=MAX_SEQUENCE_LENGTH, 
            padding='post', 
            truncating='post'
        )
        
        print(f" Данные подготовлены. Формы:")
        print(f"   - X_train_pad: {X_train_pad.shape}")
        print(f"   - X_test_pad: {X_test_pad.shape}")
        
        # 4. Генерация предсказаний
        print("\n⚡ Генерация предсказаний на сохраненной модели...")
        y_pred_train_lstm = lstm_model.predict(X_train_pad, verbose=1).flatten()
        y_pred_test_lstm = lstm_model.predict(X_test_pad, verbose=1).flatten()
        
        # 5. Расчет метрик
        print("\n📊 Расчет метрик...")
        train_mae_lstm = mean_absolute_error(y_train, y_pred_train_lstm)
        test_mae_lstm = mean_absolute_error(y_test, y_pred_test_lstm)
        
        train_rmse_lstm = np.sqrt(mean_squared_error(y_train, y_pred_train_lstm))
        test_rmse_lstm = np.sqrt(mean_squared_error(y_test, y_pred_test_lstm))
        
        train_r2_lstm = r2_score(y_train, y_pred_train_lstm)
        test_r2_lstm = r2_score(y_test, y_pred_test_lstm)
        
        training_time = 130 * 60  # 130 минут в секундах
        print(f"  Время обучения (известно из эксперимента): {training_time/60:.1f} минут")
        
        # 7. Создание объекта результатов
        results_lstm = {
            'model_name': 'lstm',
            'r2_train': train_r2_lstm,
            'r2_test': test_r2_lstm,
            'mae_train': train_mae_lstm,
            'mae_test': test_mae_lstm,
            'rmse_train': train_rmse_lstm,
            'rmse_test': test_rmse_lstm,
            'training_time': training_time,
            'params': f'LSTM: vocab_size={min(20000, len(tokenizer.word_index)+1)}, max_len={MAX_SEQUENCE_LENGTH}',
            'position': len(all_results) + 1
        }
        
        # 8. Вывод результатов
        print(f"\n ВОССТАНОВЛЕННЫЕ РЕЗУЛЬТАТЫ LSTM:")
        print(f"{'='*50}")
        print(f"TRAIN METRICS:")
        print(f"   R²  : {train_r2_lstm:.4f}")
        print(f"   MAE : {train_mae_lstm:.4f}")
        print(f"   RMSE: {train_rmse_lstm:.4f}")
        print(f"\nTEST METRICS:")
        print(f"   R²  : {test_r2_lstm:.4f}")
        print(f"   MAE : {test_mae_lstm:.4f}")
        print(f"   RMSE: {test_rmse_lstm:.4f}")
        print(f"\n Время обучения: {training_time/60:.1f} минут")
        
        # 9. Визуализация для LSTM
        plt.figure(figsize=(12, 5))
        
        plt.subplot(1, 2, 1)
        plt.scatter(y_train, y_pred_train_lstm, alpha=0.5, color='green', s=10)
        plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2)
        plt.xlabel('Реальные значения')
        plt.ylabel('Предсказанные значения')
        plt.title(f'LSTM Train (R² = {train_r2_lstm:.3f})')
        
        plt.subplot(1, 2, 2)
        plt.scatter(y_test, y_pred_test_lstm, alpha=0.5, color='red', s=10)
        plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
        plt.xlabel('Реальные значения')
        plt.ylabel('Предсказанные значения')
        plt.title(f'LSTM Test (R² = {test_r2_lstm:.3f})')
        
        plt.tight_layout()
        plt.savefig('results/lstm_restored_predictions_scatter.png')
        plt.close()
        print(" Визуализация LSTM сохранена в 'results/lstm_restored_predictions_scatter.png'")
        
        print(" LSTM результаты успешно восстановлены!")
        
    except Exception as e:
        print(f" Ошибка при восстановлении LSTM результатов: {e}")
        import traceback
        traceback.print_exc()
        results_lstm = None


🔍 ВОССТАНОВЛЕНИЕ LSTM РЕЗУЛЬТАТОВ (БЕЗ ФАЙЛА ИСТОРИИ)
✅ Все необходимые файлы найдены

📥 Загрузка токенизатора...
📥 Загрузка LSTM модели...
✅ LSTM модель успешно загружена

🔧 Подготовка данных для предсказаний...
✅ Данные подготовлены. Формы:
   - X_train_pad: (23700, 200)
   - X_test_pad: (5925, 200)

⚡ Генерация предсказаний на сохраненной модели...
741/741 ━━━━━━━━━━━━━━━━━━━━ 94s 125ms/step
186/186 ━━━━━━━━━━━━━━━━━━━━ 24s 128ms/step

📊 Расчет метрик...
⏱️  Время обучения (известно из эксперимента): 130.0 минут

📈 ВОССТАНОВЛЕННЫЕ РЕЗУЛЬТАТЫ LSTM:
TRAIN METRICS:
   R²  : 0.9014
   MAE : 0.0707
   RMSE: 0.1093

TEST METRICS:
   R²  : 0.7299
   MAE : 0.1194
   RMSE: 0.1793

⏱️  Время обучения: 130.0 минут
✅ Визуализация LSTM сохранена в 'results/lstm_restored_predictions_scatter.png'
✅ LSTM результаты успешно восстановлены!


In [ ]:
# ═════════════════════════════════════════════════════════════════════════════════
# БЛОК 10: СВОДНЫЙ АНАЛИЗ РЕЗУЛЬТАТОВ
# ═════════════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print(" СВОДНЫЙ АНАЛИЗ РЕЗУЛЬТАТОВ ВСЕХ МОДЕЛЕЙ")
print("="*70)

# Сбор всех результатов
all_results = [
    results_tfidf_gb,
    results_tfidf_rf,
    results_tfidf_ridge,
    results_tfidf_svr,
    results_char_gb,
    results_lemma_gb,
    results_hashing_gb,
    results_hashing_ridge
]

# Добавляем LSTM если он был обучен
if 'results_lstm' in globals() and results_lstm is not None:
    all_results.append(results_lstm)
    print(" LSTM результаты включены в сравнение")

# Добавляем CatBoost если он был обучен
if 'results_catboost' in globals() and results_catboost is not None:
    all_results.append(results_catboost)
    print("CatBoost результаты включены в сравнение")

# Создание DataFrame с результатами
results_df = pd.DataFrame(all_results)

# Сортировка по тестовому R² (основная метрика для регрессии)
results_df = results_df.sort_values('r2_test', ascending=False).reset_index(drop=True)

# Добавление позиции в рейтинге
results_df['position'] = range(1, len(results_df) + 1)

# Красивое форматирование для вывода
print("\n" + "="*70)
print("РЕЙТИНГ МОДЕЛЕЙ ПО TEST R² (ОСНОВНАЯ МЕТРИКА)")
print("="*70)

# Вывод топ-5 моделей
top_n = min(5, len(results_df))
for i in range(top_n):
    row = results_df.iloc[i]
    position = row['position']
    model_name = row['model_name'].replace('_', ' ').title()
    r2_test = row['r2_test']
    mae_test = row['mae_test']
    rmse_test = row['rmse_test']
    
    # Эмодзи в зависимости от позиции
    emoji = '🥇' if position == 1 else '🥈' if position == 2 else '🥉' if position == 3 else '🔸'
    
    print(f"\n{emoji} ПОЗИЦИЯ #{position}: {model_name}")
    print(f"   Test R²  : {r2_test:.4f} {'✅' if r2_test > 0.5 else '⚠️'}")
    print(f"   Test MAE : {mae_test:.4f} (чем меньше, тем лучше)")
    print(f"   Test RMSE: {rmse_test:.4f}")
    
    # Анализ переобучения
    overfit_gap = row['r2_train'] - row['r2_test']
    if overfit_gap > 0.1:
        print(f"    Внимание: переобучение (разрыв R²: {overfit_gap:.3f})")
    elif overfit_gap < 0:
        print(f"    Хорошо: нет переобучения (разрыв R²: {overfit_gap:.3f})")
    else:
        print(f"    Оптимально: небольшое переобучение (разрыв R²: {overfit_gap:.3f})")

# Полная таблица для сохранения
print(f"\n Сохранение полной таблицы результатов...")
results_df.to_csv('results/models_full_comparison.csv', index=False)
results_df.to_excel('results/models_full_comparison.xlsx', index=False)
print(f"Таблица сохранена в 'results/models_full_comparison.csv'")

# Визуализация: сравнение всех моделей
plt.figure(figsize=(15, 10))

# График 1: Сравнение R²
plt.subplot(2, 2, 1)
plt.barh(range(len(results_df)), results_df['r2_test'], color='skyblue', alpha=0.8)
plt.yticks(range(len(results_df)), [name.replace('_', ' ').title() for name in results_df['model_name']])
plt.xlabel('Test R²')
plt.title('Сравнение моделей по Test R²')
plt.grid(True, alpha=0.3)

# Добавление значений на столбцы
for i, v in enumerate(results_df['r2_test']):
    plt.text(v + 0.01, i, f'{v:.3f}', va='center')

# График 2: Сравнение MAE
plt.subplot(2, 2, 2)
plt.barh(range(len(results_df)), results_df['mae_test'], color='salmon', alpha=0.8)
plt.yticks(range(len(results_df)), [name.replace('_', ' ').title() for name in results_df['model_name']])
plt.xlabel('Test MAE')
plt.title('Сравнение моделей по Test MAE (чем меньше, тем лучше)')
plt.grid(True, alpha=0.3)

for i, v in enumerate(results_df['mae_test']):
    plt.text(v + 0.005, i, f'{v:.3f}', va='center')

# График 3: Анализ переобучения
plt.subplot(2, 2, 3)
overfit_gap = results_df['r2_train'] - results_df['r2_test']
colors = ['red' if gap > 0.1 else 'green' if gap < 0 else 'orange' for gap in overfit_gap]
plt.barh(range(len(results_df)), overfit_gap, color=colors, alpha=0.8)
plt.yticks(range(len(results_df)), [name.replace('_', ' ').title() for name in results_df['model_name']])
plt.xlabel('Разрыв R² (Train - Test)')
plt.title('Анализ переобучения')
plt.axvline(x=0, color='black', linestyle='--', alpha=0.5)
plt.grid(True, alpha=0.3)

for i, v in enumerate(overfit_gap):
    plt.text(v + 0.01, i, f'{v:.3f}', va='center')

# График 4: Сравнение RMSE
plt.subplot(2, 2, 4)
plt.scatter(results_df['mae_test'], results_df['rmse_test'], 
           s=results_df['r2_test'] * 1000,  # Размер точек по R²
           alpha=0.7, c=range(len(results_df)), cmap='viridis')
plt.xlabel('Test MAE')
plt.ylabel('Test RMSE')
plt.title('Сравнение MAE vs RMSE (размер точек = R²)')
plt.grid(True, alpha=0.3)

for i, row in results_df.iterrows():
    plt.annotate(row['model_name'].replace('_', ' ')[:10], 
                (row['mae_test'], row['rmse_test']),
                xytext=(5, 5), textcoords='offset points')

plt.tight_layout()
plt.savefig('results/models_comprehensive_comparison.png', bbox_inches='tight', dpi=300)
plt.close()

# Определение лучшей модели
best_model = results_df.iloc[0]
best_model_name = best_model['model_name']
print(f"\n" + "="*70)
print(f" ЛУЧШАЯ МОДЕЛЬ: {best_model_name.replace('_', ' ').title()}")
print("="*70)
print(f"   Test R²  : {best_model['r2_test']:.4f}")
print(f"   Test MAE : {best_model['mae_test']:.4f}")
print(f"   Test RMSE: {best_model['rmse_test']:.4f}")
print(f"   Переобучение (разрыв R²): {(best_model['r2_train'] - best_model['r2_test']):.4f}")

# Рекомендации по использованию
print(f"\n РЕКОМЕНДАЦИИ:")
if best_model_name.startswith('lstm'):
    print("   - Используйте LSTM для максимального качества")
    print("   - Требует GPU для быстрого инференса")
    print("   - Хорошо работает с длинными текстами")
elif 'gb' in best_model_name or 'catboost' in best_model_name:
    print("   - Используйте Gradient Boosting для баланса скорости и качества")
    print("   - Работает быстро на CPU")
    print("   - Легко интерпретировать важность признаков")
elif 'ridge' in best_model_name:
    print("   - Используйте Ridge для максимально быстрой работы")
    print("   - Идеально для production с ограничениями по времени")
    print("   - Стабильные предсказания без переобучения")
elif 'rf' in best_model_name:
    print("   - Используйте Random Forest для устойчивости к выбросам")
    print("   - Хорошо работает с разнообразными признаками")
    print("   - Требует больше памяти, чем GB")

# Сохранение лучшей модели как основной
best_model_path = f'results/{best_model_name}_model.pkl'
if os.path.exists(best_model_path):
    final_best_path = 'results/best_model.pkl'
    import shutil
    shutil.copy2(best_model_path, final_best_path)
    print(f"\nЛучшая модель скопирована в: {final_best_path}")
    print(f"   (оригинал: {best_model_path})")

print(f"\n ВСЕ ЭКСПЕРИМЕНТЫ ЗАВЕРШЕНЫ!")
print(f"Все результаты сохранены в директории 'results/'")
print(f"Полный отчет: 'results/models_full_comparison.csv'")
print(f"Визуализация: 'results/models_comprehensive_comparison.png'")


🏆 СВОДНЫЙ АНАЛИЗ РЕЗУЛЬТАТОВ ВСЕХ МОДЕЛЕЙ
✅ LSTM результаты включены в сравнение
CatBoost результаты включены в сравнение

РЕЙТИНГ МОДЕЛЕЙ ПО TEST R² (ОСНОВНАЯ МЕТРИКА)

🥇 ПОЗИЦИЯ #1: Char Gb
   Test R²  : 0.8686 ✅
   Test MAE : 0.0877 (чем меньше, тем лучше)
   Test RMSE: 0.1250
    Оптимально: небольшое переобучение (разрыв R²: 0.048)

🥈 ПОЗИЦИЯ #2: Hashing Gb
   Test R²  : 0.8021 ✅
   Test MAE : 0.1185 (чем меньше, тем лучше)
   Test RMSE: 0.1535
    Оптимально: небольшое переобучение (разрыв R²: 0.043)

🥉 ПОЗИЦИЯ #3: Catboost
   Test R²  : 0.8016 ✅
   Test MAE : 0.1179 (чем меньше, тем лучше)
   Test RMSE: 0.1536
    Оптимально: небольшое переобучение (разрыв R²: 0.024)

🔸 ПОЗИЦИЯ #4: Tfidf Gb
   Test R²  : 0.8007 ✅
   Test MAE : 0.1188 (чем меньше, тем лучше)
   Test RMSE: 0.1540
    Оптимально: небольшое переобучение (разрыв R²: 0.042)

🔸 ПОЗИЦИЯ #5: Lemma Gb
   Test R²  : 0.7739 ✅
   Test MAE : 0.1240 (чем меньше, тем лучше)
   Test RMSE: 0.1640
    Оптимально: небольшое переоб